# Kaggle Jupyter Notebook

This notebook has only as personal use to fetch and download the data from the challenge: https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/data.

### SETUP 

**in your .env file:**
--------------------

In [1]:
# KAGGLE_USERNAME= #YOUR USERNAME HERE
# KAGGLE_API_TOKEN= #YOUR TOKEN HERE


### PYTHON INSTALL
--------------------
Install the Kaggle API, dotenv for environment variables, and pandas<br>


In [2]:
!pip install -q kaggle python-dotenv pandas

### AUTHENTIFICATION:

In [3]:
import os
from dotenv import load_dotenv

# 1. Load variables from .env
load_dotenv()

# 2. Get variables using the EXACT uppercase names from your .env
kaggle_user = os.environ.get('KAGGLE_USERNAME')
kaggle_key = os.environ.get('KAGGLE_API_TOKEN')

# 3. Check if they were found before setting them
if not kaggle_user or not kaggle_key:
    raise ValueError("Credentials not found! Check that your .env is saved and loaded correctly.")

# 4. Map to exactly what the Kaggle API expects under the hood
os.environ['KAGGLE_USERNAME'] = kaggle_user
os.environ['KAGGLE_KEY'] = kaggle_key

# 5. Import and authenticate
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

print("Kaggle Authentication Successful!")

Kaggle Authentication Successful!


### Data setup and directory.

In [4]:
comp_name = "rsna-knee-abnormality-detection"
data_dir = "data"

# 1. Create the data/ folder
os.makedirs(data_dir, exist_ok=True)

print("Available files in this Kaggle competition:")
# 2. Fetch the file list
files_response = api.competition_list_files(comp_name)

# 3. Safely handle the API response object
try:
    # If the object has a nested 'files' attribute (newer Kaggle API versions)
    if hasattr(files_response, 'files'):
        for f in files_response.files:
            print(f" - {getattr(f, 'name', f)}")
    # If it is a standard list (older Kaggle API versions)
    elif isinstance(files_response, list):
        for f in files_response:
            print(f" - {getattr(f, 'name', f)}")
    else:
        # Fallback: just print the raw response
        print(files_response)
except Exception as e:
    print(f"Could not format file list: {e}")

Available files in this Kaggle competition:
 - sample_submission.csv
 - test.csv
 - test_series.csv
 - test_series/1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191/1.2.826.0.1.3680043.8.498.11580656442259111255675562605155903947/1.2.826.0.1.3680043.8.498.10492923471392639089206565125595901837.dcm
 - test_series/1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191/1.2.826.0.1.3680043.8.498.11580656442259111255675562605155903947/1.2.826.0.1.3680043.8.498.10813088847157507017654677978881920168.dcm
 - test_series/1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191/1.2.826.0.1.3680043.8.498.11580656442259111255675562605155903947/1.2.826.0.1.3680043.8.498.10823532752386915456266439830023207944.dcm
 - test_series/1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191/1.2.826.0.1.3680043.8.498.11580656442259111255675562605155903947/1.2.826.0.1.3680043.8.498.10900781597370949799648805888840803105.dcm
 - test_series/1.2.826.0.1.3680043.8.498.100

### DOWNLOAD - DO NOT INTERUPT

In [5]:
import time

print("Downloading train.csv...")
# Add a try-except to handle rate limits gracefully
try:
    api.competition_download_file(comp_name, "train.csv", path=data_dir)
    time.sleep(2) # Pause for 2 seconds

    print("Downloading test.csv...")
    api.competition_download_file(comp_name, "test.csv", path=data_dir)
    time.sleep(2) # Pause for 2 seconds

    # Unzip any downloaded CSVs (if they came as .zip)
    for item in os.listdir(data_dir):
        if item.endswith('.zip'):
            file_path = os.path.join(data_dir, item)
            with zipfile.ZipFile(file_path, 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            os.remove(file_path)

    print("\nCSVs Downloaded and Extracted!")

except Exception as e:
    print(f"Error downloading CSVs: {e}")
    print("If you see a 429 error, wait 5 minutes and try again.")


# --- Fetching exactly 58 Labeled Examples ---
try:
    df_train_temp = pd.read_csv(os.path.join(data_dir, "train.csv"))

    # Get 58 unique labeled study IDs
    sample_58_studies = df_train_temp['study_id'].head(58).tolist()

    print("\nDownloading 58 specific labeled studies (this will take a moment to avoid rate limits)...")

    success_count = 0
    for study_id in sample_58_studies:
        # Assuming images are in a folder named train_images
        file_path = f"train_images/{study_id}"
        try:
            api.competition_download_file(comp_name, file_path, path=data_dir)
            success_count += 1
            # Crucial: Sleep for 1.5 seconds between each folder to avoid the 429 error
            time.sleep(1.5)
        except Exception as e:
            # If a specific folder isn't found or errors out, we skip it
            pass

    print(f"\nSuccessfully fetched {success_count} labeled items.")
except Exception as e:
    print(f"Could not parse 58 labeled items yet. Make sure train.csv downloaded successfully. Error: {e}")

Error downloading CSVs: 429 Client Error: Too Many Requests for url: https://www.kaggle.com/api/v1/competitions/data/download/rsna-knee-abnormality-detection/train.csv
If you see a 429 error, wait 5 minutes and try again.
Could not parse 58 labeled items yet. Make sure train.csv downloaded successfully. Error: name 'pd' is not defined


In [6]:
# SAMPLE DATA
# Load train.csv
train_path = os.path.join(data_dir, "train.csv")
if os.path.exists(train_path):
    df_train = pd.read_csv(train_path)
    print(f"\n--- train.csv head (Total Rows: {len(df_train)}) ---")
    display(df_train.head())
else:
    print("train.csv not found.")

# Load test.csv
test_path = os.path.join(data_dir, "test.csv")
if os.path.exists(test_path):
    df_test = pd.read_csv(test_path)
    print(f"\n--- test.csv head (Total Rows: {len(df_test)}) ---")
    display(df_test.head())
else:
    print("test.csv not found.")

train.csv not found.
test.csv not found.
